<a href="https://colab.research.google.com/github/Shadowspot/comfyui/blob/main/comfyui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE = "/content/drive/MyDrive/comfyui"
custom_paths = {
    "checkpoints": "models/ckpt",
    "loras": "models/loras",
    "vae": "models/vae",
    "controlnet": "models/controlnet",
    "embedding":"embeddings",
    "workflows": "workflows",
    "output": "output"
}
def setup_custom_links(drive_root, path_dict):
    comfy_root = "/root/comfy/ComfyUI"

    for folder_type, sub_path in path_dict.items():
        drive_target = os.path.join(drive_root, sub_path)

        if folder_type == "workflows":
            local_source = os.path.join(comfy_root, "user/default/workflows")
        else:
            local_source = os.path.join(comfy_root, "models" if folder_type != "output" else "", folder_type if folder_type != "workflows" else "")

        # Ensure the target directory in Drive exists
        os.makedirs(drive_target, exist_ok=True)

        # Remove existing symlink or directory before creating a new one
        if os.path.islink(local_source):
            !rm {local_source}
        elif os.path.exists(local_source):
            !rm -rf {local_source}

        # Create the symbolic link
        !ln -s {drive_target} {local_source}
        print(f"✅ 已套用: {folder_type} -> {drive_target}")

setup_custom_links(DRIVE_BASE, custom_paths)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 已套用: checkpoints -> /content/drive/MyDrive/comfyui/models/ckpt
✅ 已套用: loras -> /content/drive/MyDrive/comfyui/models/loras
✅ 已套用: vae -> /content/drive/MyDrive/comfyui/models/vae
✅ 已套用: controlnet -> /content/drive/MyDrive/comfyui/models/controlnet
✅ 已套用: embedding -> /content/drive/MyDrive/comfyui/embeddings
✅ 已套用: workflows -> /content/drive/MyDrive/comfyui/workflows
✅ 已套用: output -> /content/drive/MyDrive/comfyui/output


In [2]:
!pip install comfy-cli

In [3]:
!yes | comfy install --nvidia

ComfyUI is already installed at the specified path: /root/comfy/ComfyUI

If you want to restore dependencies, add the '--restore' option.


In [4]:
#!wget --content-disposition -O "/root/comfy/ComfyUI/models/checkpoints/noobaiXLNAIXL_vPred10Version.safetensors" "https://huggingface.co/misri/noobaiXLNAIXL_vPred10Version/resolve/main/noobaiXLNAIXL_vPred10Version.safetensors"

In [5]:
import os

deb_file = "cloudflared-linux-amd64.deb"

# Check if the deb file already exists before downloading
if not os.path.exists(deb_file):
    print(f"Downloading {deb_file}...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/{deb_file}
else:
    print(f"{deb_file} already exists. Skipping download.")

!dpkg -i {deb_file}

cloudflared-linux-amd64.deb already exists. Skipping download.
(Reading database ... 121856 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) over (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...


In [6]:
!mkdir -p /root/comfy/ComfyUI/user/default/workflows
!curl -L -o /root/comfy/ComfyUI/user/default/workflows/示例.json https://raw.githubusercontent.com/4evergr8/ComfyColab/refs/heads/main/%E7%A4%BA%E4%BE%8B.json

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  5438  100  5438    0     0  16072      0 --:--:-- --:--:-- --:--:-- 16088


In [7]:
!nohup cloudflared tunnel --url http://127.0.0.1:8188 --protocol http2 > output.log 2>&1 &

In [8]:
!rm -rf /root/comfy/ComfyUI/output
!ln -s /content/drive/MyDrive /root/comfy/ComfyUI/output

In [13]:
!nohup python3 /root/comfy/ComfyUI/main.py --listen 0.0.0.0 --port 8188 > comfyui_output.log 2>&1 &